# Case 6 — The Pet Classifier Audit
## Is the model looking at the right thing?

A pet-photo service uses a deep-learning classifier to distinguish **cats** from **dogs**. Its held-out accuracy is high, and its explanation heatmaps often look convincing.

But the audit question is harder:

> **Does the evidence support the claim that the model is relying on the pet itself, for the reasons the heatmap seems to suggest?**

This case separates four ideas that are easy to confuse:

- **correctness** — is the prediction right?
- **confidence** — how strongly does the model score the prediction?
- **localization** — does an attribution map fall on the animal?
- **faithfulness** — does the explanation track behavior of the actual learned model?

### Learning objectives

By the end of the case, you should be able to:

- interpret gradient, Grad-CAM, and Integrated Gradients explanations;
- explain why preprocessing is part of the model being audited;
- distinguish visually plausible heatmaps from tested explanation claims;
- evaluate baseline dependence, localization, perturbation sensitivity, and parameter dependence;
- report mixed or negative explanation findings without forcing a dramatic conclusion.

### Audit rule

A heatmap is **evidence to investigate**, not a conclusion by itself.

## 1 · Can we trust the classifier enough to audit it?

### Prediction contract

The experiment uses a controlled subset of the **Oxford-IIIT Pet** dataset.

- fixed seed: `2026`
- per breed: 30 training, 10 development, 10 reserved test images
- total: **1,110 train / 370 development / 370 test**
- ImageNet-pretrained **ResNet-18** backbone remains frozen
- only a class-balanced binary logistic head is fitted
- no hyperparameter search is performed in the notebook

Because breeds are sampled equally but the dataset has more dog breeds than cat breeds, the binary class distribution is imbalanced.

> **Predict first:** What accuracy could a classifier achieve by always predicting dog? Why is balanced accuracy important here?

In [ ]:
from pathlib import Path
import sys
import json, hashlib, inspect
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
# Find the existing Case 6 package folder robustly.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in candidates if (p / 'audit.py').exists() and (p / 'plots.py').exists()),
    None
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find audit.py and plots.py. Keep this notebook inside the extracted Case 6 package."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import audit as a
import plots
a.setup()
summary=json.loads((ROOT/'outputs/summary.json').read_text())
weights=ROOT/'weights/resnet18-f37072fd.pth'
assert hashlib.sha256(weights.read_bytes()).hexdigest()==summary['weights_sha256']
rows={s:pd.read_csv(ROOT/f'cache/{s}_manifest.csv') for s in ['train','dev','test']}
for s,t in [('train','dev'),('train','test'),('dev','test')]:
    assert not set(rows[s].name)&set(rows[t].name)
display(pd.DataFrame({s:r.y.value_counts().reindex([0,1]) for s,r in rows.items()}).rename(index={0:'Cat',1:'Dog'}))
print('Dog-only test accuracy:', summary['majority_test'])
print('Torch:',torch.__version__,'| CPU threads:',torch.get_num_threads())

### Fit only the classifier head

The cached arrays contain actual 512-dimensional ResNet features computed from images. Reusing them makes the lab fast, but it means the notebook does not regenerate every embedding on every run.

The reserved test set is still untouched at this stage.

> **Interpretation checkpoint:** If development accuracy is high, what does that establish? What does it still *not* tell us about the model's reasons?

In [ ]:
net=a.PetNet(weights)
net.fit(np.load(ROOT/'cache/train_clean.npy'),rows['train'].y)
dev,pred=a.evaluate(net,np.load(ROOT/'cache/dev_clean.npy'),rows['dev'])
display(pd.DataFrame([{k:v for k,v in dev.items() if k!='confusion_matrix'}]))
display(pd.DataFrame(dev['confusion_matrix'],index=['True cat','True dog'],columns=['Predicted cat','Predicted dog']))
assert np.isclose(dev['accuracy'],summary['development']['accuracy'])

## 2 · Choose examples before looking at explanations

Explanation examples should not be selected because their heatmaps look attractive.

The selection rule is fixed first:

- for each species, choose the most confident correct example;
- also choose the most confident error;
- if no error exists, choose the least-confident remaining example;
- break ties by filename.

This produces four diagnostic examples from development predictions.

> **Before viewing them:** Which is more informative for an explanation audit — only correct examples, or a mixture of correct and incorrect predictions?

In [ ]:
selected=a.select(pred)
assert selected.name.tolist()==pd.read_csv(ROOT/'outputs/selected.csv').name.tolist()
display(selected[['name','y','prediction','p_dog','correct']])
fig,axes=plt.subplots(1,4,figsize=(13,3.5))
for ax,r in zip(axes,selected.itertuples()):
    s=np.load(ROOT/f'samples/{r.name}.npz');ax.imshow(s['rgb'][0].transpose(1,2,0));ax.axis('off')
    ax.set_title(f'{r.name}\nTrue {r.y} / predicted {r.prediction}',fontsize=8)
plt.tight_layout();plt.show()

## 3 · What input did the model actually see?

Before explaining a prediction, inspect the **preprocessed input**.

A center crop can remove information that was visible in the original photograph. The notebook therefore includes an exploratory development-only comparison with aspect-preserving resize and mean padding.

This follow-up changes framing, scale, and padding together, so it **does not isolate one causal factor**. It is a diagnostic reminder:

> An explanation describes the model applied to its processed input — not necessarily the original photograph as a human saw it.

The primary model and reserved-test protocol remain unchanged.

In [ ]:
plots.preprocessing();plt.show()
display(pd.read_csv(ROOT/'outputs/preprocessing_check.csv').round(4))

### Pause and interpret

If a prediction changes substantially under a reasonable preprocessing alternative, then a visually convincing explanation from one preprocessing pipeline should be interpreted cautiously.

This is not yet an attribution-method failure. It is evidence that **the model's effective input matters**.

## 4 · What quantity are we explaining?

The model produces a dog-vs-cat log-odds margin \(z(x)\).

We explain:

- \(+z(x)\) when the fixed target is **dog**;
- \(-z(x)\) when the fixed target is **cat**.

The target class is fixed to the **original predicted class** during interventions. If we changed the target after a prediction flip, we would silently change the question.

Three explanation methods are used:

- **Input gradient** — local sensitivity of the target margin to pixel changes;
- **Grad-CAM** — positive spatial evidence in the final convolutional block;
- **Integrated Gradients (IG)** — attribution relative to a chosen reference image.

Different methods answer different questions. They should not be treated as interchangeable "importance scores."

In [ ]:
r=selected.iloc[0];sample=np.load(ROOT/f'samples/{r["name"]}.npz')
x=torch.tensor(sample['rgb']);mask=sample['mask'];target=int(r.prediction)
with torch.no_grad():p_dog=torch.sigmoid(net(x)).item()
assert abs(p_dog-float(r.p_dog))<2e-5
gradient=a.gradient(net,x,target);cam=a.gradcam(net,x,target)[0,0].numpy()
print('Image:',r['name'],'| fixed target:',target,'| p(dog):',p_dog)
print('Grad-CAM passed the exact global-average-pooling + linear-head algebra check.')
print(inspect.getsource(a.gradcam))

## 5 · The heatmaps look convincing — should we believe them?

First inspect the maps without making a faithfulness claim.

Heatmap colors are scaled independently for visualization, so color intensity should **not** be compared across images as if it were a common numerical scale.

> **Student task:** Choose the most convincing-looking explanation. Write down what you are tempted to claim about the model.

Then ask:

> **What experiment could disconfirm that visual story?**

In [ ]:
plots.maps_figure();plt.show()

## 6 · Does the explanation depend on arbitrary choices?

### Integrated Gradients baseline dependence

IG explains the prediction **relative to a baseline**. Changing the reference image can change the attribution.

The implementation also checks the IG completeness relation numerically rather than assuming it holds:

\[
\sum_i IG_i \approx F(x) - F(x')
\]

where \(x'\) is the baseline.

The notebook increases the number of integration steps when necessary and records residuals rather than silently accepting a poor numerical approximation.

> **Predict first:** If two plausible IG baselines produce noticeably different spatial rankings, what does that imply about interpreting either map as *the* explanation?

In [ ]:
a.unit_test()
for label,baseline in a.references(x).items():
    attribution,diagnostic=a.checked_ig(net,x,baseline,target)
    print(label,{k:v for k,v in diagnostic.items() if k!='attempts'})
diagnostics=pd.DataFrame(json.loads((ROOT/'outputs/attributions.json').read_text()))
display(diagnostics[diagnostics.method.str.startswith('IG')][['name','method','steps','method_integration','score_delta','residual','passes']])
plots.signed_figure();plt.show()
_,maps=plots.data()
display(pd.DataFrame([{'name':n,'baseline_rank_correlation':a.corr(m['ig_mean'],m['ig_blur'])} for n,m in maps.items()]))

### Interpretation checkpoint

Baseline dependence is not automatically a bug. It is part of what baseline-relative attribution means.

The correct conclusion is not:

> "IG is invalid."

It is:

> "The explanation claim must state the reference against which importance is defined."

## 7 · Does the heatmap actually localize the animal?

Oxford-IIIT Pet provides segmentation trimaps.

For this audit:

- foreground = animal;
- background = non-animal;
- uncertain boundary pixels are excluded.

For each attribution map, compare:

- **foreground attribution share** — how much attribution mass lies on the animal;
- **animal area share** — how much of the evaluated image is animal;
- **enrichment** — foreground attribution share divided by animal area share.

This matters because a large animal can receive a large attribution share even from a nearly uniform map.

> **Important:** Localization is useful evidence, but it is **not ground-truth reasoning**.

In [ ]:
plots.localization();plt.show()
display(diagnostics[['name','method','foreground_share','area_share','enrichment']].round(3))

### Read enrichment, not only overlap

A heatmap can visually overlap the pet and still show little enrichment beyond the pet's image area.

Ask:

1. Does the explanation concentrate on the animal more than area alone would predict?
2. Do the three methods agree?
3. Is localization sufficient to claim faithfulness?

The answer to the third question is **no**. We still need behavioral tests.

## 8 · If highlighted regions matter, does changing them change the score?

We now challenge the visual story with controlled removals.

The experiment changes:

- top 10% of Grad-CAM pixels;
- bottom 10%;
- ten random pixel sets of the same size;
- full foreground;
- full background.

Both mean-fill and blur-fill variants are considered.

The outcome is change in the **original target margin**.

### Limitations

These are sensitivity tests, not perfect causal interventions:

- replacements create out-of-distribution images;
- foreground and background regions have unequal size;
- equal pixel count does not imply equal spatial geometry.

> **Predict first:** If Grad-CAM is behaviorally informative, what relationship should you expect among top-attribution, bottom-attribution, and random removals?

In [ ]:
fresh=pd.DataFrame(a.removal(net,x,target,mask,cam))
display(fresh[~fresh.region.str.startswith('random')].round(3))
plots.removal_figure();plt.show()

### Interpretation checkpoint

A visually concentrated map becomes more persuasive when altering its highlighted region changes the target score more than comparable low-attribution or random regions.

But perturbation results also depend on **how the replacement is constructed**. Mean-fill and blur-fill may lead to different conclusions.

So perturbation supports or challenges a claim; it does not create a perfect ground-truth explanation.

## 9 · Does the explanation depend on the learned model?

A faithful explanation should respond to the model's learned parameters.

We perform a **parameter-randomization sanity check**:

- randomize only the classifier head;
- randomize the full network;
- repeat across seeds 11, 22, and 33;
- keep the image and original target fixed.

Spatial Spearman correlation compares the original and randomized explanation rankings.

Constant or all-zero maps have undefined correlation and are reported as missing rather than falsely treated as zero similarity.

> **Reasoning question:** If an explanation remains nearly unchanged after the learned model is destroyed, what would that imply?

In [ ]:
sanity=pd.read_csv(ROOT/'outputs/randomization.csv')
display(sanity.groupby('stage')[['gradient_spearman','cam_spearman']].agg(['mean','min','max','count']))
print('Zero CAM maps:',int(sanity.cam_zero.sum()),'of',len(sanity))
plots.sanity_figure();plt.show()

### What this test can and cannot establish

If the explanation changes strongly after parameter randomization, that is evidence that it depends on the learned model.

That still does **not** prove perfect faithfulness.

Architectural priors, input edges, preprocessing, and the attribution method can all influence the resulting map.

## 10 · Can we deliberately offer the model a shortcut?

A second classifier head is fitted using images with an artificial marker:

- a 32×32 square in the top-left corner;
- red for cats;
- blue for dogs;
- 100% correlated with the training label;
- about 2% of input area.

We compare both the original clean head and the marker-trained head under:

- **clean** images;
- **aligned** markers;
- **swapped** markers.

> **Predict first:** What result would demonstrate marker dominance? What result would show only limited marker sensitivity?

In [ ]:
marked=a.PetNet(weights)
marked.fit(np.load(ROOT/'cache/train_aligned.npy'),rows['train'].y)
results=[]
for label,model in [('clean',net),('marker_trained',marked)]:
    for condition in ['clean','aligned','swapped']:
        m,_=a.evaluate(model,np.load(ROOT/f'cache/dev_{condition}.npy'),rows['dev'])
        results.append(dict(model=label,condition=condition,**m))
display(pd.DataFrame(results).drop(columns='confusion_matrix').round(4))
plots.marker_performance();plt.show()

### Negative results are still results

This shortcut challenge is deliberately artificial. Providing a shortcut does **not** guarantee that the model will rely on it strongly.

If the swapped-marker condition causes only a modest performance change, the correct conclusion is:

> **This experiment did not establish shortcut dominance.**

Do not rewrite a weak effect into a dramatic story just because the experiment was designed to look for one.

## 11 · Only now look at the reserved test set

The model and diagnostic protocol were fixed before scoring this reserved subset.

This is a **370-image subset**, not the full Oxford-IIIT Pet benchmark.

The audit also does not establish robustness to:

- external domains;
- near duplicates beyond the recorded exact-byte check;
- pretraining overlap;
- adversarial examples.

Held-out test performance supports predictive competence on this evaluation set. It does not establish explanation faithfulness.

In [ ]:
test,tp=a.evaluate(net,np.load(ROOT/'cache/test_clean.npy'),rows['test'])
assert np.isclose(test['accuracy'],summary['test']['accuracy'])
display(pd.DataFrame([{k:v for k,v in test.items() if k!='confusion_matrix'}]))
display(pd.DataFrame(test['confusion_matrix'],index=['True cat','True dog'],columns=['Predicted cat','Predicted dog']))
print('Exact JPEG-byte duplicates across selected splits:',summary['cross_split_exact_duplicates'])

## 12 · Audit verdict

Return to the stakeholder question:

> **Is the model looking at the right thing?**

Separate the evidence:

| Claim | What would support it? |
|---|---|
| **Correctness** | Held-out predictive performance |
| **Confidence** | Strength of the model's score |
| **Localization** | Attribution concentration on the animal beyond area share |
| **Behavioral relevance** | Score changes under targeted perturbations |
| **Model dependence** | Explanation changes after parameter randomization |
| **Shortcut robustness** | Response to aligned and swapped marker conditions |

No single row answers the whole question.

### Final student task

Write a **150–200 word audit conclusion** that includes:

1. what held-out performance establishes — and what it does not;
2. one numerical observation that challenges a purely visual interpretation;
3. one limitation of the baseline or perturbation tests;
4. whether the marker experiment demonstrated **dominance**, **limited sensitivity**, or **neither**;
5. one additional experiment that could change your conclusion.

Avoid unsupported claims such as:

- "the model understands pets";
- "the heatmap proves causality";
- "the model uses only the animal."

Mixed and negative findings are valid findings.

## Research lineage

This teaching case uses established methods and a public benchmark while organizing them into a stakeholder-centered audit workflow.

Key references:

- **Oxford-IIIT Pet Dataset** — Parkhi et al.
- **Grad-CAM** — Selvaraju et al. (2017)
- **Integrated Gradients** — Sundararajan, Taly & Yan (2017)
- **Sanity Checks for Saliency Maps** — Adebayo et al. (2018)

### What this case is designed to teach

The goal is not to rank explanation methods from best to worst.

The goal is to learn a reusable audit pattern:

**predictive competence → inspect the effective input → inspect explanations → challenge method assumptions → test localization → perturb behavior → randomize parameters → test shortcuts → communicate a bounded conclusion**